In [ ]:
# Описание: восстановлен простой вариант BounceStrategy без проверок маржи
class BounceStrategy(Strategy):
    level_method = 0
    buffer_atr_s = 0.5
    buffer_atr_r = 0.5
    sl_atr_s = 2.0
    sl_atr_r = 2.0
    tp_atr_s = 2.0
    tp_atr_r = 2.0
    long_only = False
    short_only = False
    max_open_positions = 3
    max_lot_size = 0.1
    lot_units = 100_000
    max_position_size = max_lot_size * lot_units
    def init(self):
        df = self.data.df
        self._i = 0
        self._close = df['Close'].to_numpy()
        self._atr = df['ATR'].to_numpy()
        self._sup = {c: df[c].to_numpy() for c in SUP_COLS[self.level_method]}
        self._res = {c: df[c].to_numpy() for c in RES_COLS[self.level_method]}
        self._open_positions = 0
    def _nearest_support(self,i,close):
        best=-np.inf
        for arr in self._sup.values():
            v=arr[i];
            if v<=close and v>best: best=v
        return best if best>-np.inf else None
    def _nearest_resistance(self,i,close):
        best=np.inf
        for arr in self._res.values():
            v=arr[i];
            if v>=close and v<best: best=v
        return best if best<np.inf else None
    def _cap_size(self,size): return min(max(1,int(round(size))), int(self.max_position_size))
    def next(self):
        i = self._i; self._i += 1
        atr = self._atr[i]
        if not np.isfinite(atr) or atr<=0: return
        close = self._close[i]
        if self._open_positions >= self.max_open_positions: return
        if not self.short_only:
            sup = self._nearest_support(i, close)
            if sup is not None:
                entry = sup + self.buffer_atr_s * atr
                risk = self.sl_atr_s * atr
                if risk > 0:
                    size = max(1, int(round((CONFIG['cash']*0.02)/risk)))
                    size = self._cap_size(size)
                    if close>entry:
                        self.buy(size=size, limit=entry, sl=entry-self.sl_atr_s*atr, tp=entry+self.tp_atr_s*atr)
                        self._open_positions += 1
        if not self.long_only:
            res = self._nearest_resistance(i, close)
            if res is not None:
                entry = res - self.buffer_atr_r * atr
                risk = self.sl_atr_r * atr
                if risk > 0:
                    size = max(1, int(round((CONFIG['cash']*0.02)/risk)))
                    size = self._cap_size(size)
                    if close<entry:
                        self.sell(size=size, limit=entry, sl=entry+self.sl_atr_r*atr, tp=entry-self.tp_atr_r*atr)
                        self._open_positions += 1